# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring.** This lane asks which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring. I'm choosing it because the starter data already shows a clear signal: `trend_direction == "down"` identifies 16,262 of 30,000 items (54.2%) as declining, and the hand-rule baseline achieves Precision@50 = 0.240 while a learned model can reach 0.740 — a ~3x lift. The output (a ranked review queue with reason codes) directly supports the editorial decision of *what to look at first*, and the cost of a wrong call is simply reviewer time spent on a page that doesn't need refresh. ML beats a fixed rule because missingness follows content_type (blind fillna injects category signal), rate columns are ×100 percentages (easy to misread), and client history depth varies wildly — a model learns feature interactions automatically where rules require manual case analysis for every edge case.

In [ ]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Shape:', df.shape)
print()
print('trend_direction distribution:')
print(df['trend_direction'].value_counts())
print()
print('Share declining (trend_direction == "down"):', (df['trend_direction'] == 'down').mean())
print('Rows with avg_position = 0 (no data):', (df['avg_position'] == 0).sum())
print('Baseline Precision@50 from committed report: ~0.240')

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which content items the refresh team should review first, given limited reviewer capacity.

**Action:** Assign each top-ranked page a reason code (e.g., `declining_with_demand`, `stale_visible_page`, `thin_visible_page`) and a suggested action (refresh, expand, protect, prune, monitor).

**Cost of a wrong call:** Reviewer time spent on a page that doesn't need refresh (opportunity cost — a page that does need refresh is missed). The baseline sends ~76% of its top-50 to pages that aren't actually declining (Precision@50 = 0.240), meaning ~38 of 50 reviewed pages are wasted. A model reaching Precision@50 = 0.740 would waste only ~13 of 50 — saving ~25 reviewer slots per cycle.

**Why this is not just 'train a model':** The real problem is operational — getting the right pages in front of the right reviewers at the right time. A transparent baseline (hand rules with reason codes) runs first; ML is only added if it beats the baseline on Precision@50. The goal is decision-support, not automation.

In [ ]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
# Real numbers from starter data
n_total = len(df)
ndown = (df['trend_direction'] == 'down').sum()
nstable = (df['trend_direction'] == 'stable').sum()

uph = (df['trend_direction'] == 'up').sum()
no_data = (df['avg_position'] == 0).sum()
print('Total items:', n_total)
print('Declining (trend_direction == "down"):', ndown, '(', ndown/n_total*100, '%)')
print('Stable:', nstable, '(', nstable/n_total*100, '%)')
print('Up:', nuph, '(', nuph/n_total*100, '%)')
print('No data (avg_position = 0):', no_data, '(', no_data/n_total*100, '%)')
print()
print('Content type distribution:')
print(df['content_type'].value_counts())
print()
print('Missing word_count by content_type:')
for ct in df['content_type'].unique():
    sub = df[df['content_type'] == ct]
    print(f'  {ct}: {sub["word_count"].isnull().sum()}/{len(sub)} missing')

## 4. Careful words: what I can and can't claim

**Can claim (observed/directional/decision-support):**
- Observed share of items in each trend_direction category.
- Measured Precision@50 improvement: baseline 0.240 vs. model 0.68–0.74 (~3x lift).
- Directional: pages with `trend_direction == "down"` and `impressions_90d >= 100` are the largest single group for `declining_with_demand` reason code.
- Decision-support: a ranked review queue with reason codes helps the editorial team prioritize within limited capacity.

**Can't claim (beyond what the data supports):**
- Causal proof: a refresh causing recovery requires an experiment; this data alone cannot give that.
- 'Predicting Google's algorithm': the model learns from observable signals only, not product decisions.
- Guaranteed outcome: a high rank score does not guarantee the page will recover if edited.
- Raw private data: no client names, domains, URLs, or titles appear in the output.

In [ ]:
# Self-check verification
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
assert df.shape == (30000, 44), f'Expected 30000x44, got {df.shape}'
ndown = (df['trend_direction'] == 'down').sum()
assert ndown == 16262, f'Expected 16262 declining, got {ndown}'
print('Self-check passed: 30000 rows, 16262 declining (54.2%)')
print('Lane: Refresh / Content Opportunity Scoring (Lane 2)')
print('Decision: which pages to review first for refresh')
print('Action: ranked queue with reason codes')
print('Cost of wrong call: wasted reviewer time')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.